# Notebook 01 — Data Loading & QA Dataset (Kaggle T4)

Uses Kaggle Input feature (no download needed - dataset is already on Kaggle's servers).

**Setup:**
1. Click **+ Add Input** in right sidebar
2. Search for: `raddar/chest-xrays-indiana-university`
3. Click **Add**

**Secrets needed:** `GROQ_API_KEY`, `HF_TOKEN`

In [ ]:
!pip install -q groq tqdm pandas

In [ ]:
import os, sys, subprocess, glob

WORKING_DIR = '/kaggle/working'

# Auto-detect Kaggle input directory (handles any naming)
def find_dataset_dir():
    kaggle_input = '/kaggle/input'
    if not os.path.exists(kaggle_input):
        return None
    for entry in os.listdir(kaggle_input):
        candidate = os.path.join(kaggle_input, entry)
        if os.path.isdir(candidate):
            # Check if it has the indiana CSV files
            csv_files = glob.glob(os.path.join(candidate, '**', 'indiana_reports.csv'), recursive=True)
            if csv_files:
                return os.path.dirname(csv_files[0])
    return None

INPUT_DIR = find_dataset_dir()
if not INPUT_DIR:
    raise RuntimeError(
        'Dataset not found in /kaggle/input/\n'
        'Click "+ Add Input" in right sidebar and add "raddar/chest-xrays-indiana-university"'
    )

print(f'✓ Dataset found at {INPUT_DIR}')
print(f'\nContents:')
for f in os.listdir(INPUT_DIR):
    print(f'  {f}')

In [ ]:
# Get API keys from Kaggle secrets
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret('GROQ_API_KEY')
HF_TOKEN = user_secrets.get_secret('HF_TOKEN')

print('✓ Secrets loaded')

In [ ]:
# Clone repo
REPO_PATH = os.path.join(WORKING_DIR, 'cxr-rag-system')
if not os.path.exists(REPO_PATH):
    print('Cloning repository...')
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mohamedtaha77/cxr-rag-system.git', REPO_PATH], check=True)
else:
    print('Updating repository...')
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

sys.path.insert(0, REPO_PATH)
print('✓ Repository ready')

In [ ]:
# Auto-detect image directory in Kaggle input
import glob

png_files = glob.glob(os.path.join(INPUT_DIR, '**', '*.png'), recursive=True)
if not png_files:
    raise RuntimeError(f'No PNG images found in {INPUT_DIR}')

IMAGES_DIR = os.path.dirname(png_files[0])
print(f'Found {len(png_files)} PNG images in: {IMAGES_DIR}')

In [ ]:
# Load dataset from Kaggle CSVs (in /kaggle/input/)
from src.data.openi_loader import OpenILoader

loader = OpenILoader(images_dir=IMAGES_DIR)
df = loader.load_from_kaggle_csvs(kaggle_dir=INPUT_DIR)

print(f'Loaded {len(df)} studies')
print(df.head(3))

In [ ]:
# Train/val/test split
import pandas as pd

train_df, val_df, test_df = loader.train_val_test_split(df)
full_df = pd.concat([train_df, val_df, test_df])

corpus_path = os.path.join(WORKING_DIR, 'reports_corpus.csv')
full_df.to_csv(corpus_path, index=False)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Saved: {corpus_path}')

In [ ]:
# QA Generation via Groq
from src.data.qa_creator import QACreator

creator = QACreator(groq_api_key=GROQ_API_KEY)
QA_OUTPUT = os.path.join(WORKING_DIR, 'qa_dataset.jsonl')

pairs = creator.generate_dataset(
    df=full_df,
    output_path=QA_OUTPUT,
    max_studies=200,  # ~3,000 pairs, ~30 min
)

print(f'Generated {len(pairs)} QA pairs')

In [ ]:
# Stats and samples
import json

qa_df = pd.read_json(QA_OUTPUT, lines=True)
print(f'Total pairs   : {len(qa_df)}')
print(f'Unique studies: {qa_df["study_id"].nunique()}')
print(f'\nSplit:\n{qa_df["split"].value_counts()}')
print(f'\nCategory:\n{qa_df["category"].value_counts()}')

print('\n=== Sample QA pairs ===')
for s in qa_df.head(3).to_dict('records'):
    print(f"Q: {s['question']}")
    print(f"A: {s['answer']}\n")

print(f'\n✓ All outputs in {WORKING_DIR}')